
# FairWarn-SHS — Strict Early-Warning Feature Ablation

This notebook tests whether the system can still identify at-risk students
after removing outcome-adjacent academic score variables:

- Q10_MathsScore
- Q11_EnglishScore
- Q12_CAScore

It reruns Logistic Regression, Random Forest, Feature-only MLP, and all-edge GraphSAGE
across five fixed seeds.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import random
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, balanced_accuracy_score,
    accuracy_score, brier_score_loss
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"
SEEDS = [42, 123, 456, 789, 1010]

REMOVED_FEATURES = [
    "Q10_MathsScore",
    "Q11_EnglishScore",
    "Q12_CAScore",
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def metrics(y_true, prob, pred):
    return {
        "AUC_ROC": roc_auc_score(y_true, prob),
        "AUC_PR": average_precision_score(y_true, prob),
        "Precision_AtRisk": precision_score(y_true, pred, zero_division=0),
        "Recall_AtRisk": recall_score(y_true, pred, zero_division=0),
        "F1_AtRisk": f1_score(y_true, pred, zero_division=0),
        "Weighted_F1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "Brier_Score": brier_score_loss(y_true, prob),
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

excluded = {
    "Node_ID", "Roster_Code", "School_Code", "Class_Code",
    "Label_Available", "TARGET_AtRisk", *REMOVED_FEATURES
}

strict_features = [c for c in nodes.columns if c not in excluded]

labelled_df = nodes.loc[labelled_mask].copy()
X_labelled = labelled_df[strict_features].copy()
y_labelled = labelled_df["TARGET_AtRisk"].astype(int).to_numpy()

print("Removed features:", REMOVED_FEATURES)
print("Strict feature count:", len(strict_features))
print("Labelled students:", len(labelled_df))
print("At-risk:", int((y_labelled == 1).sum()))
print("Not-at-risk:", int((y_labelled == 0).sum()))


In [ ]:

def make_preprocessor(frame):
    numeric = frame.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [c for c in frame.columns if c not in numeric]

    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])

baseline_rows = []

for seed in SEEDS:
    X_train, X_test, y_train, y_test = train_test_split(
        X_labelled, y_labelled,
        test_size=0.20,
        stratify=y_labelled,
        random_state=seed
    )

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=seed
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=500, class_weight="balanced",
            random_state=seed, n_jobs=-1
        ),
        "Feature-only MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=seed
        ),
    }

    for model_name, model in models.items():
        pipe = Pipeline([
            ("preprocess", make_preprocessor(X_train)),
            ("model", model)
        ])

        pipe.fit(X_train, y_train)
        prob = pipe.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)

        row = {
            "Seed": seed,
            "Model": model_name,
            **metrics(y_test, prob, pred)
        }
        baseline_rows.append(row)

        print(
            f"{model_name} | seed {seed} | "
            f"AUC-PR={row['AUC_PR']:.4f} | "
            f"Recall={row['Recall_AtRisk']:.4f} | "
            f"F1={row['F1_AtRisk']:.4f}"
        )

baseline_metrics_df = pd.DataFrame(baseline_rows)


In [ ]:

# Build strict-feature graph.
X_all = nodes[strict_features].copy()

numeric = X_all.select_dtypes(include=[np.number]).columns.tolist()
categorical = [c for c in X_all.columns if c not in numeric]

for c in numeric:
    X_all[c] = X_all[c].fillna(X_all[c].median())

for c in categorical:
    mode = X_all[c].mode(dropna=True)
    X_all[c] = X_all[c].fillna(mode.iloc[0] if not mode.empty else "Missing")

graph_preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])

X_graph = graph_preprocessor.fit_transform(X_all).astype(np.float32)
y_all = nodes["TARGET_AtRisk"].fillna(-1).astype(int).to_numpy()

node_map = {node_id: i for i, node_id in enumerate(nodes["Node_ID"])}
pairs = []

for _, row in edges.iterrows():
    s_id = row["Source_Node_ID"]
    t_id = row["Target_Node_ID"]
    if s_id in node_map and t_id in node_map:
        s, t = node_map[s_id], node_map[t_id]
        pairs.extend([(s, t), (t, s)])

edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

graph_data = Data(
    x=torch.tensor(X_graph, dtype=torch.float32),
    edge_index=edge_index,
    y=torch.tensor(y_all, dtype=torch.long)
)

print("Nodes:", graph_data.num_nodes)
print("Labelled nodes:", int(labelled_mask.sum()))
print("Undirected edges:", edge_index.shape[1] // 2)


In [ ]:

def make_masks(seed):
    idx = np.where(labelled_mask)[0]
    y_lab = y_all[idx]

    train_val, test = train_test_split(
        idx, test_size=0.20, stratify=y_lab, random_state=seed
    )

    train_val_y = y_all[train_val]

    train, val = train_test_split(
        train_val,
        test_size=0.1875,
        stratify=train_val_y,
        random_state=seed
    )

    masks = []
    for part in [train, val, test]:
        mask = torch.zeros(len(y_all), dtype=torch.bool)
        mask[part] = True
        masks.append(mask)

    return masks

class StrictGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, 64, aggr="mean")
        self.conv2 = SAGEConv(64, 32, aggr="mean")
        self.classifier = torch.nn.Linear(32, 2)
        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


In [ ]:

graph_rows = []

for seed in SEEDS:
    set_seed(seed)
    train_mask, val_mask, test_mask = make_masks(seed)

    graph = graph_data.clone()
    graph.train_mask = train_mask
    graph.val_mask = val_mask
    graph.test_mask = test_mask
    graph = graph.to(device)

    model = StrictGraphSAGE(graph.num_node_features).to(device)

    train_labels = graph.y[graph.train_mask]
    counts = torch.bincount(train_labels, minlength=2).float()
    weights = (counts.sum() / (2.0 * counts.clamp_min(1.0))).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.005, weight_decay=5e-4
    )

    best_state = None
    best_val_ap = -np.inf
    best_epoch = 0
    wait = 0

    for epoch in range(1, 501):
        model.train()
        optimizer.zero_grad()

        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(
            logits[graph.train_mask],
            graph.y[graph.train_mask],
            weight=weights
        )
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(graph.x, graph.edge_index)
            prob = torch.softmax(logits, dim=1)[:, 1]
            val_true = graph.y[graph.val_mask].cpu().numpy()
            val_prob = prob[graph.val_mask].cpu().numpy()
            val_ap = average_precision_score(val_true, val_prob)

        if val_ap > best_val_ap + 1e-6:
            best_val_ap = val_ap
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= 40:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)

    truth = graph.y[graph.test_mask].cpu().numpy()
    test_prob = prob[graph.test_mask].cpu().numpy()
    test_pred = pred[graph.test_mask].cpu().numpy()

    result = {
        "Seed": seed,
        "Model": "Strict GraphSAGE",
        "Best_Epoch": best_epoch,
        **metrics(truth, test_prob, test_pred)
    }
    graph_rows.append(result)

    print(
        f"Strict GraphSAGE | seed {seed} | "
        f"AUC-PR={result['AUC_PR']:.4f} | "
        f"Recall={result['Recall_AtRisk']:.4f} | "
        f"F1={result['F1_AtRisk']:.4f}"
    )

graph_metrics_df = pd.DataFrame(graph_rows)


In [ ]:

all_metrics_df = pd.concat(
    [baseline_metrics_df, graph_metrics_df],
    ignore_index=True
)

metric_columns = [
    "AUC_ROC", "AUC_PR", "Precision_AtRisk",
    "Recall_AtRisk", "F1_AtRisk", "Weighted_F1",
    "Balanced_Accuracy", "Accuracy", "Brier_Score"
]

summary_rows = []

for model_name, group in all_metrics_df.groupby("Model"):
    row = {
        "Model": model_name,
        "Seeds": group["Seed"].nunique()
    }

    for metric in metric_columns:
        row[f"{metric}_Mean"] = group[metric].mean()
        row[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_rows.append(row)

strict_summary_df = pd.DataFrame(summary_rows).sort_values(
    "AUC_PR_Mean", ascending=False
).reset_index(drop=True)

strict_summary_df


In [ ]:

# Compare with previously established full-feature means.
reference = pd.DataFrame([
    {"Model": "Logistic Regression", "Full_AUC_PR": 0.9332, "Full_Recall": 0.8666, "Full_F1": 0.8379},
    {"Model": "Random Forest", "Full_AUC_PR": 0.9556, "Full_Recall": 0.8457, "Full_F1": 0.8762},
    {"Model": "Feature-only MLP", "Full_AUC_PR": 0.9200, "Full_Recall": 0.7583, "Full_F1": 0.8119},
    {"Model": "Strict GraphSAGE", "Full_AUC_PR": 0.8850, "Full_Recall": 0.8000, "Full_F1": 0.7787},
])

comparison_df = strict_summary_df.merge(reference, on="Model", how="left")

comparison_df["Delta_AUC_PR_Strict_Minus_Full"] = (
    comparison_df["AUC_PR_Mean"] - comparison_df["Full_AUC_PR"]
)
comparison_df["Delta_Recall_Strict_Minus_Full"] = (
    comparison_df["Recall_AtRisk_Mean"] - comparison_df["Full_Recall"]
)
comparison_df["Delta_F1_Strict_Minus_Full"] = (
    comparison_df["F1_AtRisk_Mean"] - comparison_df["Full_F1"]
)

comparison_df


In [ ]:

plt.figure(figsize=(9, 5))
plot_df = strict_summary_df.sort_values("AUC_PR_Mean", ascending=False)

plt.bar(
    plot_df["Model"],
    plot_df["AUC_PR_Mean"],
    yerr=plot_df["AUC_PR_SD"],
    capsize=4
)

plt.ylabel("Strict early-warning AUC-PR")
plt.title("Strict early-warning model comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()



## Interpretation rule

The purpose is not to force the strict models to outperform the full-feature models.
The purpose is to measure how much predictive performance remains when direct
academic score variables are removed.

Any performance loss must be reported honestly.


In [ ]:

strict_summary_df.to_csv(
    "fairwarn09_strict_model_summary_mean_sd.csv",
    index=False
)

all_metrics_df.to_csv(
    "fairwarn09_strict_model_metrics_by_seed.csv",
    index=False
)

comparison_df.to_csv(
    "fairwarn09_strict_vs_full_feature_comparison.csv",
    index=False
)

files.download("fairwarn09_strict_model_summary_mean_sd.csv")
files.download("fairwarn09_strict_model_metrics_by_seed.csv")
files.download("fairwarn09_strict_vs_full_feature_comparison.csv")
